# YOLO26 GPU Benchmark — PCB 裸板瑕疵偵測（Colab T4）

在 Colab T4 GPU 上量測三種後端的推論延遲與精度：PyTorch FP32（`.pt` 基準）、TensorRT FP16、TensorRT INT8。搭配本機 `scripts/benchmark_cpu.py` 量到的 ONNX Runtime CPU 數字，組成 `reports/benchmark.md` 完整的後端對照表（Phase 2 步驟 2.4，詳見 repo 的 `plan.md`）。

**這個 notebook 必須用 T4**（不是訓練時建議的 L4/A100）：benchmark 數字的可比性與誠實聲明都依賴固定、可重現的硬體規格——換更快的 GPU 會讓這裡量到的數字失去對照意義，之後也沒辦法老實寫進報告。

**使用前**：
1. `Runtime → Change runtime type → GPU → T4`（**不要選 L4/A100** — 下面有一格會檢查並警告，但選對比較省事）
2. 確認 Phase 1 的 grouped 訓練已完成，`best.pt` 還在 Google Drive（預期路徑：`/content/drive/MyDrive/pcb_defect/artifacts/yolo26s_pcb_grouped_640/best.pt`）
3. `Run all`；中途會跳出 Google Drive 授權視窗，同意即可

**預估時間**：15–25 分鐘（TensorRT INT8 校準最花時間，需要跑一遍校準資料）。

**跑完之後**：最後一格會印出一段用 `=== COPY ... ===` 圍起來的 JSON —— 把那整段複製貼回聊天視窗給 Claude 就好，不需要下載或搬動任何檔案（這次沒有大檔案，TensorRT engine 是裝置綁定的產物，量完即丟，不會進 Google Drive 或 Hugging Face）。

In [ ]:
# ============================================================
# Config — 執行前先確認/修改這裡
# ============================================================

REPO_URL = "https://github.com/tun0000/pcb-defect-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/pcb_defect"
RUN_NAME = "yolo26s_pcb_grouped_640"  # 對應 train_colab.ipynb 的 grouped run
DATA_ROOT = "/content/datasets/pcb"
SEED = 42

N_IMAGES = 100
N_CYCLES = 2  # -> 200 次計時推論，與本機 scripts/benchmark_cpu.py 一致
WARMUP = 30
CONF = 0.25       # 計時用的顯示門檻，與 verify_onnx_parity.py / demo 一致
VAL_CONF = 0.001  # 算 mAP 用的門檻，與 evaluate.py / export_models.py 一致

print(f"run_name={RUN_NAME}  data_root={DATA_ROOT}")

In [ ]:
# 只裝 ultralytics，不動 Colab 預裝的 torch（版本已針對 Colab GPU 優化）
!pip install -q ultralytics==8.4.89

import ultralytics

ultralytics.checks()
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0)
print(f"GPU: {gpu_name}")

if "T4" not in gpu_name:
    print(
        f"\n*** 警告：目前配到的是 {gpu_name}，不是 T4！ ***\n"
        "這個 notebook 的目的是量測 T4 的基準數字，才能跟本機 CPU 數字放進同一張誠實對照表。\n"
        "建議：Runtime -> Change runtime type -> GPU -> T4，換卡後重新 Run all。\n"
        "如果繼續在這張卡上跑，reports/benchmark.md 的 device 欄位要老實標成這張卡，不能寫 T4。"
    )
else:
    print("OK: T4 GPU 確認。")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os

if not os.path.isdir("/content/repo"):
    !git clone {REPO_URL} /content/repo
else:
    !git -C /content/repo pull

# base dependencies only（kagglehub/pillow/pyyaml）— 不裝 [train] extra，
# 不動 Colab 自己的 torch/ultralytics 安裝
!pip install -q -e /content/repo

In [ ]:
!python -m pcb_defect.data_prep.prepare --out {DATA_ROOT} --strategy grouped --seed {SEED}

In [ ]:
import json
from pathlib import Path

with open(f"{DATA_ROOT}/conversion_report.json", encoding="utf-8") as f:
    conversion_report = json.load(f)

assert conversion_report["images_total"] == 693, conversion_report
assert conversion_report["boxes_total"] == 2953, conversion_report

data_yaml = f"{DATA_ROOT}/data.yaml"
test_paths_all = sorted(Path(f"{DATA_ROOT}/images/test").glob("*.jpg"))
print(f"OK: 693 images / 2953 boxes - 與本機 weights/grouped/best.pt 同一份切分（同 seed={SEED}）")
print(f"test split: {len(test_paths_all)} images at {DATA_ROOT}/images/test")

## Benchmark 方法學

跟本機 `scripts/benchmark_cpu.py` 用同一套規則，三個後端的數字才能放進同一張表：

- 固定同一批 100 張 test 圖（`grouped` split，同 seed=42 —— 跟本機、跟 `verify_onnx_parity.py`、跟 `export_models.py` 的 fidelity 檢查用的是同一份切分）
- 每個後端：先 warmup 30 次（不計時），再循環 2 輪 = 200 次計時推論、batch=1
- `time.perf_counter()` 計**端到端**（前處理＋推論＋後處理），呼叫前後都插 `torch.cuda.synchronize()` —— GPU 呼叫是非同步的，沒有這道 barrier 計出來的時間會偏低、不真實
- 每個後端都在 test split 上單獨跑一次 `model.val(conf=0.001)`（跟 `export_models.py` 一致），量化「這個後端犧牲了多少精度換速度」——FP16/INT8 的 FPS 領先必須對照它的 mAP50-95 掉了多少分才有意義

In [ ]:
import time

import numpy as np
from PIL import Image
from ultralytics import YOLO

test_paths = sorted(Path(f"{DATA_ROOT}/images/test").glob("*.jpg"))[:N_IMAGES]
if len(test_paths) < N_IMAGES:
    print(f"WARNING: only {len(test_paths)} test images available (wanted {N_IMAGES})")
images = [Image.open(p).convert("RGB").copy() for p in test_paths]
print(f"loaded {len(images)} test images into RAM")


def time_backend(model) -> dict:
    """Warmup(30) then 200 synchronized end-to-end timed predict() calls."""
    for i in range(WARMUP):
        model.predict(source=images[i % len(images)], imgsz=640, conf=CONF, device=0, verbose=False)
    torch.cuda.synchronize()

    n_runs = len(images) * N_CYCLES
    latencies_ms = np.empty(n_runs, dtype=np.float64)
    for i in range(n_runs):
        image = images[i % len(images)]
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        model.predict(source=image, imgsz=640, conf=CONF, device=0, verbose=False)
        torch.cuda.synchronize()
        latencies_ms[i] = (time.perf_counter() - t0) * 1000

    p50 = float(np.percentile(latencies_ms, 50))
    p95 = float(np.percentile(latencies_ms, 95))
    return {
        "n_runs": n_runs,
        "p50_ms": round(p50, 3),
        "p95_ms": round(p95, 3),
        "mean_ms": round(float(latencies_ms.mean()), 3),
        "fps_from_p50": round(1000.0 / p50, 2),
    }


def fidelity_val(weights_path) -> dict:
    """mAP50 / mAP50-95 on the test split, conf=0.001 (matches scripts/export_models.py)."""
    m = YOLO(weights_path)
    metrics = m.val(
        data=data_yaml, split="test", imgsz=640, conf=VAL_CONF, iou=0.7, plots=False, verbose=False
    )
    return {"map50": float(metrics.box.map50), "map5095": float(metrics.box.map)}


all_results = {}

## 後端 1／3 — PyTorch FP32（`.pt` 基準）

In [ ]:
print("=== PyTorch FP32 (.pt) ===")
pt_weights = f"{DRIVE_ROOT}/artifacts/{RUN_NAME}/best.pt"
assert os.path.isfile(pt_weights), (
    f"{pt_weights} not found - check train_colab.ipynb's packaging cell actually ran, "
    "or that DRIVE_ROOT/RUN_NAME above match what you used for training."
)
pt_model = YOLO(pt_weights)

lat = time_backend(pt_model)
fid = fidelity_val(pt_weights)
all_results["pytorch_fp32_t4"] = {
    "backend": "pytorch",
    "precision": "fp32",
    "device": gpu_name,
    **lat,
    **fid,
}
print(f"p50={lat['p50_ms']:.2f}ms  p95={lat['p95_ms']:.2f}ms  FPS={lat['fps_from_p50']:.2f}")
print(f"mAP50={fid['map50']:.4f}  mAP50-95={fid['map5095']:.4f}")

## 後端 2／3 — TensorRT FP16

用 ultralytics 8.4.80+ 的新版 export API：`quantize=16`（取代已棄用的 `half=True`）。TensorRT engine 是裝置綁定的產物，只在這台 T4 上建立、benchmark，不會上傳 Google Drive 或 Hugging Face（`plan.md` §2.2/2.4 的既定政策 — 只有 `.pt`/`.onnx` 會上傳）。

已知的 NMS-free 限制：TensorRT < 8.5.0 會讓 `end2end` 匯出靜默退回成需要外部 NMS 的版本（不會報錯，只是不再是原本的 e2e 頭）——下面會印出實際的 TensorRT 版本並檢查。

In [ ]:
import os
import shutil

print("=== TensorRT FP16 (quantize=16) ===")
fp16_dir = "/content/trt_fp16"
os.makedirs(fp16_dir, exist_ok=True)
shutil.copy2(pt_weights, f"{fp16_dir}/best.pt")

fp16_model = YOLO(f"{fp16_dir}/best.pt")
fp16_engine_path = fp16_model.export(
    format="engine", imgsz=640, batch=1, dynamic=False, device=0, quantize=16, simplify=True
)

import tensorrt as trt

trt_version = trt.__version__
print(f"TensorRT version: {trt_version}")
if tuple(int(x) for x in trt_version.split(".")[:2]) < (8, 5):
    print(
        f"*** 警告：TensorRT {trt_version} < 8.5.0 - end2end 匯出可能已靜默退回非 e2e 版本，"
        "下面量到的延遲/後處理語意可能與 ONNX(e2e) 基準不同，reports/benchmark.md 要註記這件事。 ***"
    )

fp16_model_loaded = YOLO(fp16_engine_path)
lat = time_backend(fp16_model_loaded)
fid = fidelity_val(fp16_engine_path)
all_results["tensorrt_fp16_t4"] = {
    "backend": "tensorrt",
    "precision": "fp16",
    "device": gpu_name,
    "tensorrt_version": trt_version,
    **lat,
    **fid,
}
print(f"p50={lat['p50_ms']:.2f}ms  p95={lat['p95_ms']:.2f}ms  FPS={lat['fps_from_p50']:.2f}")
print(f"mAP50={fid['map50']:.4f}  mAP50-95={fid['map5095']:.4f}")

## 後端 3／3 — TensorRT INT8

`quantize=8`＋`data=`（校準用，內部會用 `data.yaml` 的 train split，不是 test split —— 校準不能碰之後要拿來算 fidelity 的 test 資料）＋`fraction=1.0`（用完整校準資料，這是官方預設值，不是特別加大）。

INT8 是三個後端裡最可能犧牲精度換速度的一個：如果下面 mAP50-95 掉超過 2 個百分點（跟 `scripts/export_models.py` 的 ONNX fidelity 門檻一致），`reports/benchmark.md` 會照實寫「INT8 只作為對照數字，不建議實際部署」，不會為了好看的 FPS 隱藏這件事。

In [ ]:
print("=== TensorRT INT8 (quantize=8) ===")
int8_dir = "/content/trt_int8"
os.makedirs(int8_dir, exist_ok=True)
shutil.copy2(pt_weights, f"{int8_dir}/best.pt")

int8_model = YOLO(f"{int8_dir}/best.pt")
int8_engine_path = int8_model.export(
    format="engine",
    imgsz=640,
    batch=1,
    dynamic=False,
    device=0,
    quantize=8,
    data=data_yaml,  # 校準用 train split，與 test（fidelity 評估用）分開
    fraction=1.0,
    simplify=True,
)

int8_model_loaded = YOLO(int8_engine_path)
lat = time_backend(int8_model_loaded)
fid = fidelity_val(int8_engine_path)
all_results["tensorrt_int8_t4"] = {
    "backend": "tensorrt",
    "precision": "int8",
    "device": gpu_name,
    "tensorrt_version": trt_version,
    **lat,
    **fid,
}
print(f"p50={lat['p50_ms']:.2f}ms  p95={lat['p95_ms']:.2f}ms  FPS={lat['fps_from_p50']:.2f}")
print(f"mAP50={fid['map50']:.4f}  mAP50-95={fid['map5095']:.4f}")

fp32_map5095 = all_results["pytorch_fp32_t4"]["map5095"]
delta = fid["map5095"] - fp32_map5095
if abs(delta) > 0.02:
    print(f"*** INT8 mAP50-95 與 FP32 基準差 {delta:+.4f}（超過 2 個百分點）- 報告要老實寫這個取捨 ***")

## 組報告

彙整三個後端的結果，印出可以直接複製回去的 JSON。

In [ ]:
report = {
    "hardware": {
        "gpu": gpu_name,
        "colab_tier": "Colab Pro (T4 runtime)",
        "ultralytics_version": ultralytics.__version__,
        "torch_version": torch.__version__,
        "tensorrt_version": trt_version,
    },
    "n_images": len(images),
    "n_cycles": N_CYCLES,
    "warmup": WARMUP,
    "conf": CONF,
    "val_conf": VAL_CONF,
    "results": all_results,
    "note": (
        "GPU 端計時含 torch.cuda.synchronize() barrier 的端到端量測（前處理+推論+後處理），"
        "方法學與本機 scripts/benchmark_cpu.py 一致，可直接放進同一張表。TensorRT 版本/精度"
        "與 Ultralytics 官方發布的 T4 數字所用的環境、批次大小、量測方式都不一定相同，不能"
        "直接比較。INT8 的校準資料來自 data.yaml 的 train split（與這裡拿來算 fidelity 的 "
        "test split 分開，calibration 不碰 test 資料）。"
    ),
}

print(f"\n{'backend':<12}{'precision':<10}{'p50(ms)':>9}{'p95(ms)':>9}{'FPS':>7}{'mAP50-95':>10}")
for tag, r in all_results.items():
    print(
        f"{r['backend']:<12}{r['precision']:<10}{r['p50_ms']:>9.2f}"
        f"{r['p95_ms']:>9.2f}{r['fps_from_p50']:>7.2f}{r['map5095']:>10.4f}"
    )

print("\n" + "=" * 70)
print("COPY EVERYTHING BELOW THIS LINE BACK TO CLAUDE (paste in chat, not a file)")
print("=" * 70)
print(json.dumps(report, indent=2, ensure_ascii=False))